# Stage 1 pilot — `random_data` FIM generation (random variant)
**Kaggle notebook — thin controller only. All logic lives in
`scripts/generate_fim_variants.py`, `src/fim/`.**

Buckets the curated 10,000-file sample (`sample_filtered_data_10000/`) by
AST span type and samples the **random** FIM variant from it
(`data-stage-1.md` §6), pushing the result to
`experiment/random_data/` in `the-stack-v3-python-fim-data`.

CPU-only, single in-memory pass over the already-curated 10k-file pool —
no GPU, no streaming. Estimated runtime: roughly 5–10 minutes (clone + deps
~1-2 min, `load_dataset()` of the 10k rows ~1-3 min, AST bucketing +
sampling well under a minute, upload well under a minute).

Split into one notebook per variant (this one, and its sibling
`distributed_data_10000.ipynb`) so each can run independently. The random-vs-distributed
comparison chart is produced once *both* variants exist in the repo —
whichever notebook runs second completes it automatically (see
`scripts/generate_fim_variants.py`'s `fetch_other_variant_counts`).

## Required Kaggle setup before running
| Setting | Value |
|---|---|
| Accelerator | None (CPU only) |
| Internet | **ON** |
| Secret: `HF_TOKEN` | Hugging Face **write** token |
| Secret: `WANDB_API_KEY` | Optional — W&B API key from https://wandb.ai/authorize |


In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────────
import os
import shutil
import subprocess
import sys

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Cloning branch : {BRANCH or 'main'}")
print(f"Requested commit : {COMMIT or '(latest on branch)'}")

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()
current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

print(f"\n✓ Repository ready at {REPO_DIR}")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")


In [ ]:
# ── Cell 2: Install dependencies (CPU-only) ───────────────────────────────────
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "datasets", "pyarrow", "huggingface_hub", "pyyaml", "matplotlib", "wandb", "weave",
    ],
    check=True,
)


In [ ]:
# ── Cell 3: Authenticate to Hugging Face (+ optional W&B) ─────────────────────
# HF_TOKEN is stored as a Kaggle Secret — NEVER hardcode tokens.
# Add it: Kaggle account → Settings → Secrets → Add New Secret
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

# W&B tracking is optional here — the scripts fall back to printing a
# warning and skipping W&B logging if this secret isn't set (see
# src/curation/wandb_logger.py / scripts/generate_fim_variants.py's
# init_wandb_run). Add the WANDB_API_KEY secret to enable it.
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("✓ HF_TOKEN and WANDB_API_KEY loaded from Kaggle Secrets")
except Exception:
    print("✓ HF_TOKEN loaded. WANDB_API_KEY not set — W&B tracking will be skipped.")


In [ ]:
# ── Cell 4: Generate the random variant ─────────────────────────────────
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/generate_fim_variants.py", "--variant", "random"],
    check=True,
)


In [ ]:
# ── Cell 5: Show metadata.json + the bucket-distribution chart ───────────────
from IPython.display import Image, display
import json
import os

with open(f"{REPO_DIR}/reports/experiment_random_data_metadata.json", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2))

chart_path = f"{REPO_DIR}/reports/plots/bucket_distribution.png"
if os.path.exists(chart_path):
    display(Image(filename=chart_path))
else:
    print("Comparison chart not generated yet — run the other variant's "
          "notebook too (distributed_data_10000.ipynb) to complete it."
          if "random" == "random" else
          "Comparison chart not generated yet — run the other variant's "
          "notebook too (random_data_10000.ipynb) to complete it.")

print(
    "\n" + "=" * 70 +
    "\nThis is a generation-time span-distribution check, not a winner "
    "decision. The actual random-vs-distributed winner is chosen only "
    "after training + SAFIM eval comparison on both variants "
    "(data-stage-1.md §7-8) — scripts/analyze_pilot_results.py is what "
    "makes that call, not this notebook's bucket counts." +
    "\n" + "=" * 70
)
